In [5]:
import os
import pandas as pd

# Get the current working directory (where the notebook is running)
CURRENT_DIR = os.getcwd()

# Go up one level to the project root
PROJECT_ROOT = os.path.dirname(CURRENT_DIR)

# Build the path to the data file
DATA_PATH = os.path.join(PROJECT_ROOT, "data", "train_merged.parquet")

# Load the parquet file
df = pd.read_parquet(DATA_PATH)[
    [
        # identifiers / target
        "TransactionID", "isFraud", "TransactionDT", "TransactionAmt",
        
        # product
        "ProductCD",
        
        # card info
        "card1", "card2", "card3", "card4", "card5", "card6",
        
        # address / distance
        "addr1", "addr2", "dist1", "dist2",
        
        # email
        "P_emaildomain", "R_emaildomain",
        
        # device
        "DeviceType", "DeviceInfo"
    ]
]

In [6]:
print(df.columns)

Index(['TransactionID', 'isFraud', 'TransactionDT', 'TransactionAmt',
       'ProductCD', 'card1', 'card2', 'card3', 'card4', 'card5', 'card6',
       'addr1', 'addr2', 'dist1', 'dist2', 'P_emaildomain', 'R_emaildomain',
       'DeviceType', 'DeviceInfo'],
      dtype='object')


In [7]:
na_frac = df.isna().mean()  # fraction of missing values per column
cols_to_keep = na_frac[na_frac <= 0.10].index
df = df[cols_to_keep]


In [8]:
df = df.dropna()

In [11]:
print(df.shape)
print(df.columns)

(579498, 11)
Index(['TransactionID', 'isFraud', 'TransactionDT', 'TransactionAmt',
       'ProductCD', 'card1', 'card2', 'card3', 'card4', 'card5', 'card6'],
      dtype='object')


In [12]:
print(df.head(5))

   TransactionID  isFraud  TransactionDT  TransactionAmt ProductCD  card1  \
1        2987001        0          86401            29.0         W   2755   
2        2987002        0          86469            59.0         W   4663   
3        2987003        0          86499            50.0         W  18132   
4        2987004        0          86506            50.0         H   4497   
5        2987005        0          86510            49.0         W   5937   

   card2  card3       card4  card5   card6  
1  404.0  150.0  mastercard  102.0  credit  
2  490.0  150.0        visa  166.0   debit  
3  567.0  150.0  mastercard  117.0   debit  
4  514.0  150.0  mastercard  102.0  credit  
5  555.0  150.0        visa  226.0   debit  


In [13]:
# --- 1) Drop TransactionID (index only) ---
df = df.drop(columns=["TransactionID"])

# --- 2) Derive time-based features ---
# Convert TransactionDT (seconds since reference) into hour-of-day
df["hour_of_day"] = (df["TransactionDT"] // 3600) % 24
df["day_of_week"] = (df["TransactionDT"] // (3600*24)) % 7

In [15]:
from sklearn.preprocessing import OneHotEncoder

# Categorical columns
categorical_cols = ["ProductCD", "card4", "card6"]

# One-hot encode categoricals
encoder = OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore")
encoded = encoder.fit_transform(df[categorical_cols])

# Get new column names
encoded_cols = encoder.get_feature_names_out(categorical_cols)

# Create encoded DataFrame
df_encoded = pd.DataFrame(encoded, columns=encoded_cols, index=df.index)

# Drop old categoricals and join encoded
df = df.drop(columns=categorical_cols).join(df_encoded)

# Save for later
df.to_parquet("ieee_fraud_features_encoded.parquet", index=False)

print("Final shape:", df.shape)
print("Columns:", df.columns.tolist())


Final shape: (579498, 19)
Columns: ['isFraud', 'TransactionDT', 'TransactionAmt', 'card1', 'card2', 'card3', 'card5', 'hour_of_day', 'day_of_week', 'ProductCD_H', 'ProductCD_R', 'ProductCD_S', 'ProductCD_W', 'card4_discover', 'card4_mastercard', 'card4_visa', 'card6_credit', 'card6_debit', 'card6_debit or credit']


In [2]:
import pandas as pd
import os


CURRENT_DIR = os.getcwd()

# Go up one level to the project root
PROJECT_ROOT = os.path.dirname(CURRENT_DIR)

# Build the path to the data file
DATA_PATH = os.path.join(PROJECT_ROOT, "data", "ieee_fraud_features_encoded.parquet")

df_processed = pd.read_parquet(DATA_PATH)

In [3]:
print(df_processed.columns)

Index(['isFraud', 'TransactionDT', 'TransactionAmt', 'card1', 'card2', 'card3',
       'card5', 'hour_of_day', 'day_of_week', 'ProductCD_H', 'ProductCD_R',
       'ProductCD_S', 'ProductCD_W', 'card4_discover', 'card4_mastercard',
       'card4_visa', 'card6_credit', 'card6_debit', 'card6_debit or credit'],
      dtype='object')
